In [30]:
import re
import numpy as np
import pandas as pd
from typing import List, Dict, Any, Tuple
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import f1_score

### Baseline: TF-IDF + Log Reg

In [31]:
def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"[^\w\s%&\-/\.]", " ", s, flags=re.U)
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

In [32]:
def prepare_df(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    for c in TEXT_COLS:
        if c not in df.columns:
            raise ValueError(f"Колонка '{c}' не найдена в CSV")
        df[c] = df[c].fillna("")
    if "level1" not in df.columns or "level2" not in df.columns:
        raise ValueError("Нужны колонки 'level1' и 'level2'")
    df["X"] = df[TEXT_COLS].agg(" ".join, axis=1).map(clean_text)
    return df

In [33]:
def build_pipeline() -> Pipeline:
    vect = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.9,
        sublinear_tf=True
    )
    clf = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        n_jobs=-1
    )
    return Pipeline([("tfidf", vect), ("logreg", clf)])

def train_global(df: pd.DataFrame, target: str) -> Pipeline:
    work = df.dropna(subset=[target]).copy()
    X = work["X"].values
    y = work[target].astype(str).values
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    pipe = build_pipeline()
    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_te)
    # print(f"\n=== {target.upper()} (GLOBAL) ===")
    # print(classification_report(y_te, y_pred, digits=4))
    labels_sorted = sorted(np.unique(y))


    return pipe

def train_hierarchical_level2(df: pd.DataFrame) -> Dict[str, Pipeline]:
    models = {}
    parents = sorted(df["level1"].dropna().astype(str).unique().tolist())
    for parent in parents:
        sub = df[(df["level1"].astype(str) == parent) & (~df["level2"].isna())].copy()
        if sub.empty or sub["level2"].nunique() < 5:
            print(f"[WARN] пропускаю '{parent}': мало классов level2")
            continue
        X = sub["X"].values
        y = sub["level2"].astype(str).values
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.2, random_state=SEED, stratify=y
        )
        pipe = build_pipeline()
        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_te)
        # print(f"\n=== LEVEL2 для родителя '{parent}' ===")
        # print(classification_report(y_te, y_pred, digits=4))
        labels_sorted = sorted(np.unique(y))
        models[parent] = pipe
    return models

In [34]:
class TwoLevelBaseline:
    def __init__(self, hierarchical_level2: bool = True):
        self.hier = hierarchical_level2
        self.model_level1: Pipeline | None = None
        self.model_level2_global: Pipeline | None = None
        self.model_level2_by_parent: Dict[str, Pipeline] = {}

    def fit(self, df: pd.DataFrame):
        self.model_level1 = train_global(df, "level1")

        if self.hier:
            self.model_level2_by_parent = train_hierarchical_level2(df)

        else:
            self.model_level2_global = train_global(df, "level2")


    def predict(self, df: pd.DataFrame,target_1:str,target_2:str) -> List[Dict[str, Any]]:
        texts = (df[target_1].fillna("") + " " + df[target_2].fillna("")).tolist()
        X = [clean_text(t if isinstance(t, str) else "") for t in texts]

        l1_preds = self.model_level1.predict(X)
        out: List[Dict[str, Any]] = []

        for x, l1 in zip(X, l1_preds):
            if not x:
                out.append({target_1: None, target_2: None})
                continue

            if self.hier and (l1 in self.model_level2_by_parent):
                l2 = self.model_level2_by_parent[l1].predict([x])[0]
            elif (not self.hier) and (self.model_level2_global is not None):
                l2 = self.model_level2_global.predict([x])[0]
            else:
                l2 = None

            out.append({target_1: l1, target_2: l2})

        return out


In [35]:
def F1_score(
    true_df: pd.DataFrame,
    preds,
    true_cols: Tuple[str, str] = ("level1", "level2"),
    pred_cols: Tuple[str, str] | None = None,
    drop_na: bool = True,
    average: str = "macro",
) -> float:

    pred_df = pd.DataFrame(preds)



    t1, t2 = true_cols
    p1, p2 = pred_cols


    data = pd.DataFrame({
        "true_l1": true_df[t1].astype(str),
        "true_l2": true_df[t2].astype(str),
        "pred_l1": pred_df[p1].astype(str),
        "pred_l2": pred_df[p2].astype(str),
    })

    if drop_na:
        mask = data.notna().all(axis=1) & (data["pred_l1"] != "None") & (data["pred_l2"] != "None")
        data = data[mask]


    y_true = data["true_l1"] + "__" + data["true_l2"]
    y_pred = data["pred_l1"] + "__" + data["pred_l2"]

    return f1_score(y_true, y_pred, average=average)

In [36]:
SEED = 42
HIERARCHICAL_LEVEL2 = True
TEXT_COLS = ["title", "text"]
df = prepare_df("dataset_processed.csv")
train_df,test_df=train_test_split(df,test_size=0.2,random_state=SEED)
train_df=train_df[train_df['level2'].map(df['level2'].value_counts()) > 5]

In [37]:
mdl = TwoLevelBaseline(hierarchical_level2=HIERARCHICAL_LEVEL2)
mdl.fit(train_df)

[WARN] пропускаю 'crypto': мало классов level2


In [38]:
baseline_pred=mdl.predict(test_df,'level1','level2')

In [39]:
print(f'F1 score baseline: {F1_score(test_df,baseline_pred,pred_cols=(("level1", "level2")))}')

F1 score baseline: 0.019619973486522314


### LSTM + SVM

In [40]:
pip install gensim -q

In [41]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from gensim.models import Word2Vec
from typing import List, Dict, Any

In [42]:
torch.manual_seed(42)
np.random.seed(42)

def build_texts(df: pd.DataFrame, t1="title", t2="text") -> List[str]:
    return (
        ((df[t1].fillna("") + " ") * 3)
        + ((df[t2].fillna("") + " ") * 2)
        + "Keywords: " + ((df["keywords"].fillna("") + " ") * 2)
        + "Topic: " + df["topic"].fillna("") + " "
    ).str.strip().tolist()


def pad_ids(tok_lists: List[List[str]], stoi: Dict[str, int], max_len: int = 64) -> torch.Tensor:
    arr = []
    for toks in tok_lists:
        ids = [stoi.get(t, 0) for t in toks][:max_len]
        ids += [0] * (max_len - len(ids))
        arr.append(ids)
    return torch.tensor(arr, dtype=torch.long)


def normalize_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.lower().replace("\xa0", " ").replace("&nbsp;", " ")
    s = s.replace("<br>", " ").replace("<br/>", " ").replace("\n", " ")
    return " ".join(s.split())


def tokenize(text: str) -> List[str]:
    import re
    text = text.replace("s&p 500", "s&p_500").replace("фрс сша", "фрс_сша")
    return re.findall(r"[\w$%]+", text.lower())

In [43]:
class Encoder(nn.Module):
    def __init__(self, emb_matrix: np.ndarray, hidden: int = 128, feat_dim: int = 256, dropout: float = 0.3):
        super().__init__()
        V, D = emb_matrix.shape
        self.emb = nn.Embedding(V, D, padding_idx=0)
        with torch.no_grad():
            self.emb.weight.copy_(torch.tensor(emb_matrix))
        self.lstm = nn.LSTM(D, hidden, num_layers=2, batch_first=True, bidirectional=True, dropout=dropout)
        self.fc1 = nn.Linear(hidden * 4, feat_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, ids):
        x, _ = self.lstm(self.emb(ids))
        mean_pool = x.mean(1)
        max_pool, _ = x.max(1)
        z = torch.cat([mean_pool, max_pool], 1)
        z = self.drop(F.relu(self.fc1(z)))
        return z  # [B, feat_dim]


class LSTMHieSVM:
    def __init__(self, embed_dim=200, hidden=128, feat_dim=256, max_len=64, epochs=5, batch_size=64, lr=1e-3):
        self.embed_dim = embed_dim
        self.hidden = hidden
        self.feat_dim = feat_dim
        self.max_len = max_len
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.stoi = {"<pad>": 0}
        self.encoder = None
        self.enc_l1 = LabelEncoder()
        self.svm_l1 = LinearSVC()
        self.svm_l2_by_parent: Dict[str, LinearSVC] = {}
        self.enc_l2_by_parent: Dict[str, LabelEncoder] = {}


    def _train_w2v(self, sentences: List[List[str]]):
        w2v = Word2Vec(sentences, vector_size=self.embed_dim, window=5, min_count=1, sg=1)
        for w in w2v.wv.index_to_key:
            self.stoi.setdefault(w, len(self.stoi))
        V = len(self.stoi)
        W = np.zeros((V, self.embed_dim), np.float32)
        for w, i in self.stoi.items():
            if w != "<pad>":
                W[i] = w2v.wv[w] if w in w2v.wv else np.random.normal(0, 0.05, self.embed_dim)
        return W

    def _make_mask_matrix(self, df: pd.DataFrame, l1_classes: List[str], l2_classes: List[str]):
        # matrix [n_l1, n_l2] where 1 = allowed
        mp = {p: set(df[df.level1.astype(str) == p].level2.astype(str).unique()) for p in l1_classes}
        mat = np.zeros((len(l1_classes), len(l2_classes)), np.float32)
        for i, p in enumerate(l1_classes):
            for j, c in enumerate(l2_classes):
                if c in mp[p]:
                    mat[i, j] = 1.0
        return torch.tensor(mat)


    def fit(self, df: pd.DataFrame):
        df = df.reset_index(drop=True)
        texts = build_texts(df)
        tok = [tokenize(normalize_text(t)) for t in texts]
        W = self._train_w2v(tok)
        ids = pad_ids(tok, self.stoi, self.max_len)

        y1 = self.enc_l1.fit_transform(df["level1"].astype(str).values)
        l1_classes = list(self.enc_l1.classes_)
        # для маски берём все уникальные L2
        all_l2 = sorted(df["level2"].astype(str).dropna().unique().tolist())
        l2_to_idx = {c: i for i, c in enumerate(all_l2)}

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        mask_mat = self._make_mask_matrix(df, l1_classes, all_l2).to(device)
        self.encoder = Encoder(W, hidden=self.hidden, feat_dim=self.feat_dim).to(device)
        head_l1 = nn.Linear(self.feat_dim, len(l1_classes)).to(device)
        head_l2 = nn.Linear(self.feat_dim, len(all_l2)).to(device)
        opt = torch.optim.AdamW(list(self.encoder.parameters()) + list(head_l1.parameters()) + list(head_l2.parameters()), lr=self.lr)
        ce = nn.CrossEntropyLoss()

        X_ids = ids.to(device)
        y1_t = torch.tensor(y1, dtype=torch.long, device=device)
        # y2 индексы в пространстве all_l2
        y2_t = torch.tensor([l2_to_idx[str(v)] for v in df["level2"].astype(str)], dtype=torch.long, device=device)

        n = len(df)
        for ep in range(self.epochs):
            perm = torch.randperm(n)
            for i in range(0, n, self.batch_size):
                idx = perm[i : i + self.batch_size]
                feats = self.encoder(X_ids[idx])
                logits1 = head_l1(feats)
                logits2 = head_l2(feats)
                # иерархическая маска для L2 по TRUE L1
                m = mask_mat[y1_t[idx]]  # [B, n_l2]
                logits2 = logits2.masked_fill(m <= 0, torch.finfo(logits2.dtype).min)
                loss = ce(logits1, y1_t[idx]) + ce(logits2, y2_t[idx])
                opt.zero_grad(); loss.backward(); opt.step()

        # признаки для SVM
        with torch.no_grad():
            X = self.encoder(X_ids).cpu().numpy()
        self.svm_l1.fit(X, y1)

        self.svm_l2_by_parent.clear(); self.enc_l2_by_parent.clear()
        for parent, sub in df.groupby("level1"):
            sub = sub.dropna(subset=["level2"]).copy()
            if sub["level2"].nunique() < 2:
                continue
            enc = LabelEncoder().fit(sub["level2"].astype(str))
            clf = LinearSVC().fit(X[sub.index], enc.transform(sub["level2"].astype(str)))
            self.enc_l2_by_parent[str(parent)] = enc
            self.svm_l2_by_parent[str(parent)] = clf
        return self

    def predict(self, df: pd.DataFrame) -> List[Dict[str, Any]]:
        texts = build_texts(df)
        tok = [tokenize(normalize_text(t)) for t in texts]
        ids = pad_ids(tok, self.stoi, self.max_len)
        device = next(self.encoder.parameters()).device
        with torch.no_grad():
            X = self.encoder(ids.to(device)).cpu().numpy()
        l1_idx = self.svm_l1.predict(X)
        l1_labels = self.enc_l1.inverse_transform(l1_idx)
        out = []
        for i, p in enumerate(l1_labels):
            p = str(p)
            if p in self.svm_l2_by_parent:
                l2_idx = self.svm_l2_by_parent[p].predict(X[i:i+1])[0]
                l2 = self.enc_l2_by_parent[p].inverse_transform([l2_idx])[0]
            else:
                l2 = None
            out.append({"level1": p, "level2": l2})
        return out


In [44]:
clf = LSTMHieSVM(epochs=5)
clf.fit(train_df)
SVM_preds = clf.predict(test_df)

In [45]:
print(f'F1 score LSTM + SVM: {F1_score(test_df,SVM_preds,pred_cols=("level1", "level2"))}')

F1 score LSTM + SVM: 0.03267619267619268


### BERT

In [46]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import LabelEncoder
from typing import List, Dict, Any, Tuple

torch.manual_seed(42); np.random.seed(42)

def build_texts(df: pd.DataFrame, t1="title", t2="text") -> List[str]:
    return (
        ((df[t1].fillna("") + " ") * 3)
        + df[t2].fillna("") + " "
        + "Keywords: " + ((df["keywords"].fillna("") + " ") * 2)
        + "Topic: " + df["topic"].fillna("") + " "
    ).str.strip().tolist()

class TwoLevelDataset(Dataset):
    def __init__(self, texts: List[str], tok, l1=None, l2=None, max_len: int = 256):
        enc = tok(texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        self.input_ids, self.attn = enc["input_ids"], enc["attention_mask"]
        self.l1 = None if l1 is None else torch.tensor(l1, dtype=torch.long)
        self.l2 = None if l2 is None else torch.tensor(l2, dtype=torch.long)
    def __len__(self): return self.input_ids.size(0)
    def __getitem__(self, i):
        item = {"input_ids": self.input_ids[i], "attention_mask": self.attn[i]}
        if self.l1 is not None: item["l1"] = self.l1[i]
        if self.l2 is not None: item["l2"] = self.l2[i]
        return item

class BertTwoLevel(nn.Module):
    def __init__(self, base_name: str, num_l1: int, num_l2: int, dropout: float = 0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(base_name)
        hid = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.head_l1 = nn.Linear(hid, num_l1)
        self.head_l2 = nn.Linear(hid, num_l2)
    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(out.last_hidden_state[:, 0])
        return self.head_l1(cls), self.head_l2(cls)

class BertTwoLevelClassifier:
    def __init__(self, model_name: str = "cointegrated/rubert-tiny2", hier_mask: bool = True, device: str | None = None):
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.model_name = model_name
        self.hier_mask = hier_mask
        self.enc_l1, self.enc_l2 = LabelEncoder(), LabelEncoder()
        self.parent2children: Dict[str, set] = {}

    def fit(self, df: pd.DataFrame, text_cols: Tuple[str, str] = ("title","text"),
            epochs: int = 3, batch_size: int = 16, lr: float = 2e-5, max_len: int = 256):
        df = df.reset_index(drop=True)
        texts = build_texts(df, *text_cols)
        y1 = self.enc_l1.fit_transform(df["level1"].astype(str).values)
        y2 = self.enc_l2.fit_transform(df["level2"].astype(str).values)


        for p, sub in df.groupby("level1"):
            ch = self.enc_l2.transform(sub["level2"].astype(str).unique())
            self.parent2children[str(p)] = set(map(int, ch))

        dset = TwoLevelDataset(texts, self.tok, y1, y2, max_len=max_len)
        loader = DataLoader(dset, batch_size=batch_size, shuffle=True)

        model = BertTwoLevel(self.model_name, num_l1=len(self.enc_l1.classes_), num_l2=len(self.enc_l2.classes_))
        model.to(self.device).train()

        opt = AdamW(model.parameters(), lr=lr)
        ce_l1 = nn.CrossEntropyLoss()
        ce_l2 = nn.CrossEntropyLoss()

        for _ in range(epochs):
            for batch in loader:
                ids = batch["input_ids"].to(self.device)
                attn = batch["attention_mask"].to(self.device)
                l1 = batch["l1"].to(self.device)
                l2 = batch["l2"].to(self.device)

                logits1, logits2 = model(ids, attn)
                loss = ce_l1(logits1, l1) + ce_l2(logits2, l2)

                opt.zero_grad(); loss.backward(); opt.step()

        self.model = model.eval()  # готов к инференсу

    def predict(self, df: pd.DataFrame, text_cols: Tuple[str,str] = ("title","text"),
                out_cols: Tuple[str,str] = ("level1","level2"), batch_size: int = 32, max_len: int = 256) -> List[Dict[str, Any]]:
        texts = build_texts(df, *text_cols)
        dset = TwoLevelDataset(texts, self.tok, max_len=max_len)
        loader = DataLoader(dset, batch_size=batch_size, shuffle=False)

        preds = []
        for batch in loader:
            ids = batch["input_ids"].to(self.device)
            attn = batch["attention_mask"].to(self.device)
            with torch.no_grad():
                l1_logits, l2_logits = self.model(ids, attn)
                # L1
                l1_idx = l1_logits.argmax(dim=1).cpu().numpy()
                l1_labels = self.enc_l1.inverse_transform(l1_idx)

                # L2 (с маской недопустимых детей для предсказанного L1)
                if self.hier_mask:
                    l2_masked = l2_logits.clone()
                    for i, parent in enumerate(l1_labels):
                        allow = self.parent2children.get(str(parent), None)
                        if allow is None:  # нет знаний — без маски
                            continue
                        mask = torch.ones(l2_masked.size(1), dtype=torch.bool, device=l2_masked.device)
                        mask[list(allow)] = False
                        l2_masked[i, mask] = -1e9
                    l2_idx = l2_masked.argmax(dim=1).cpu().numpy()
                else:
                    l2_idx = l2_logits.argmax(dim=1).cpu().numpy()

                l2_labels = self.enc_l2.inverse_transform(l2_idx)

                for p, c in zip(l1_labels, l2_labels):
                    preds.append({out_cols[0]: str(p), out_cols[1]: str(c)})

        return preds

In [48]:
clf = BertTwoLevelClassifier(model_name="cointegrated/rubert-tiny2", hier_mask=True)
clf.fit(train_df, text_cols=("title","text"), epochs=6, batch_size=64, lr=2e-5, max_len=256)


config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

In [ ]:
BERT_preds = clf.predict(test_df, text_cols=("title","text"), out_cols=("level1","level2"))

In [37]:
print(f'F1 score BERT: {F1_score(test_df,BERT_preds,pred_cols=("level1", "level2"))}')

F1 score BERT: 0.019731601731601732


In [51]:
test_df["level1"].head()

,level1
1448,other
2934,corporate
794,other
1029,crypto
8,other


In [56]:
# pred_df = pd.DataFrame(SVM_preds)

# result_df = pd.DataFrame({
#     "true_level1": test_df["level1"].values,
#     "true_level2": test_df["level2"].values,
#     "pred_level1": pred_df["level1"].astype(str),
#     "pred_level2": pred_df["level2"].astype(str),
# })


# result_df.to_csv("svm_predictions.csv", index=False, encoding="utf-8-sig")

# print("✅ Файл 'svm_predictions.csv' сохранён!")
# print(result_df.head())